In [ ]:
%%shell
cat > /etc/apt/sources.list.d/debian.list <<'EOF'
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster.gpg] http://deb.debian.org/debian buster main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster-updates.gpg] http://deb.debian.org/debian buster-updates main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-security-buster.gpg] http://deb.debian.org/debian-security buster/updates main
EOF

apt-key adv --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A

apt-key export 77E11517 | gpg --dearmour -o /usr/share/keyrings/debian-buster.gpg
apt-key export 22F3D138 | gpg --dearmour -o /usr/share/keyrings/debian-buster-updates.gpg
apt-key export E562B32A | gpg --dearmour -o /usr/share/keyrings/debian-security-buster.gpg


cat > /etc/apt/preferences.d/chromium.pref << 'EOF'
Package: *
Pin: release a=eoan
Pin-Priority: 500

Package: *
Pin: origin "deb.debian.org"
Pin-Priority: 300

Package: chromium*
Pin: origin "deb.debian.org"
Pin-Priority: 700
EOF

# Install chromium and chromium-driver
apt-get update
apt-get install chromium chromium-driver

# Install selenium
pip install selenium

Executing: /tmp/apt-key-gpghome.MI55Z63Wpy/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
gpg: key DCC9EFBF77E11517: public key "Debian Stable Release Key (10/buster) <debian-release@lists.debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.n20hiD4mpN/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
gpg: key DC30D7C23CBBABEE: public key "Debian Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.fhkOFOMJ1i/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A
gpg: key 4DFAB270CAA96DFA: public key "Debian Security Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64  InRelease [1,581 B

In [17]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import datetime
#Colab에선 웹브라우저 창이 뜨지 않으므로 별도 설정한다.
from selenium import webdriver
import time 

# 크롬 드라이버 경로 설정
driver_path = "chromedriver"

# 크롬 옵션 설정
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')  # 창 숨기기 모드 실행
chrome_options.add_argument('--disable-gpu')  # GPU 사용 안함
chrome_options.add_argument('--disable-dev-shm-usage')

# 크롬 드라이버 실행
driver = webdriver.Chrome(driver_path, options=chrome_options)

# 웹 페이지 열기
url = "https://puradakchicken.com/startup/store.asp"
driver.get(url)

# 모든 가게 정보 추출
store_info_list = []
while True:
    store_table = driver.find_element_by_xpath('//table[@class="table3"]')
    rows = store_table.find_elements_by_tag_name("tr")[1:]  # 테이블 헤더 제외
    for row in rows:
        columns = row.find_elements_by_tag_name("td")
        name = columns[0].text
        address = columns[1].text
        phone = columns[2].text
        store_info_list.append({"name": name, "address": address, "phone": phone})
        
    next_button = driver.find_elements_by_xpath('//a[@class="next"]')
    if len(next_button) == 0:
        # 다음 버튼이 없으면 종료
        break
    else:
        # 다음 버튼 클릭
        next_button[0].click()

# 크롬 드라이버 종료
driver.quit()

# pandas DataFrame으로 변환
store_df = pd.DataFrame(store_info_list)

# 결과 출력
print(store_df)


WebDriverException: ignored